## 🛠️ 실습 2 - 시장의 증거를 찾아라 

<p style="font-size: 16px; font-weight: normal; margin-top: 10px;">
<strong>[미션]</strong> 최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 조사하라<br>
이번 실습에서는 SERPER 내장 Tool을 연결하여 Agent가 실제 웹 검색을 수행하도록 구성합니다.
</p>
---

## 🛠️ 프로젝트 개발 환경 구축

이 노트북은 **CrewAI**와 **OpenAI**를 활용한 에이전트 개발 환경 설정을 안내합니다.  
아래 단계에 따라 가상환경을 구축하고 필요한 패키지를 설치해 주세요.


### 1. 가상환경 설정
터미널(Terminal)에서(Ctrl+Shif+`) 실행합니다.

* **가상환경 생성:** `env02`이라는 독립된 환경을 구축합니다.
    ```bash
    python -m venv env02
    ```
* **가상환경 활성화:** 생성된 환경으로 진입합니다. (Windows CMD 기준)
    ```bash
    env02\Scripts\activate
    ```

### 2. 필수 라이브러리 설치
진입한 가상환경 터미널에서 아래 명령어를 실행하여 핵심 패키지들을 설치합니다.

| 패키지 | 용도 |
| :--- | :--- |
| **`crewai`** | Agentic AI 기반 멀티 에이전트 협업 프레임워크 |
| **`crewai-tools`** | CrewAI 에이전트가 활용하는 도구 모음(웹 검색, 문서 읽기, API 호출 등) 패키지 |
| **`openai`** | GPT 모델 연동 API |
| **`python-dotenv`** | 환경 변수(.env) 로드 |


```bash
pip install crewai crewai-tools openai python-dotenv
```

### 1️⃣ 환경 변수 로드

In [ ]:
# 아래의 코드는 .env 파일에서 OPENAI_MODEL과 SERPER_API_KEY 환경 변수를 로드하는 예시입니다. 
# .env 파일에 OPENAI_MODEL과 SERPER_API_KEY가 정의되어 있어야 합니다.
# 또한, .env 파일에는 OPENAI_API_KEY도 정의되어 있어야 합니다.

import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL")
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

print(f"OPENAI_MODEL: {OPENAI_MODEL}")

### 2️⃣ SERPER Tool 연결

In [ ]:
from crewai_tools import SerperDevTool

# SERPER_API_KEY를 명시적으로 전달하여 어떤 키가 사용되는지 명확히 합니다.
search_tool = SerperDevTool(api_key=SERPER_API_KEY)
print("SERPER Tool이 성공적으로 연결되었습니다.")

### 3️⃣ Agent 정의 

In [ ]:
from crewai import Agent, LLM

llm = LLM(model=OPENAI_MODEL)

research_agent = Agent(
    role="AI 기반 데이터 분석 전략 컨설턴트",
    goal=(
        "최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 조사하고 "
        "근거 기반으로 정리한다."
    ),
    backstory=(
        "기업 디지털 전환 전략을 자문하는 전문가로서 "
        "출처 기반 리서치를 중시한다." 
        "모든 결과물은 반드시 한국어로 작성한다." 
    ),
    tools=[search_tool], # SERPER Tool을 Agent의 도구로 추가
    llm=llm,             # .env에서 로드한 모델을 명시적으로 지정
    verbose=True, 
    max_iter=3  # Tool 호출 포함 최대 반복 횟수 제한
)

### 4️⃣ Task 정의 

In [ ]:
from crewai import Task

research_task = Task(
    description=(
        "최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 최소 3개 조사하라. "
        "도입 배경, 실행 방식, 성과를 정리하라." 
        "모든 결과물은 반드시 한국어로 작성하라."
    ),
    expected_output=(
        "각 사례별로:\n"
        "1. 기업명\n"
        "2. 도입 배경\n"
        "3. 실행 방식\n"
        "4. 성과\n"
        "5. 출처(URL 포함, 불확실하면 '출처 미확인'으로 표기)\n"
        "최소 3개 사례 포함."
    ),
    agent=research_agent
)

### 5️⃣ Crew 구성

In [ ]:
from crewai import Crew

research_crew = Crew(
    agents=[research_agent],
    tasks=[research_task],
    verbose=True
)

### 6️⃣ Crew 실행과 최종 결과 출력

In [ ]:
result = research_crew.kickoff()

print("\n===== 최종 결과 =====\n")
print(result)

### 7️⃣ 결과 저장

In [ ]:
from datetime import datetime
import os

os.makedirs("outputs", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"outputs/chapter2_research_{timestamp}.md"

with open(filename, "w", encoding="utf-8") as f:
    f.write("# 실습 2 - 시장의 증거를 찾아라\n\n")
    f.write(str(result))

print(f"\n✅ 결과가 저장되었습니다:\n{filename}")

## 🔎 결과를 관찰해 보세요

1. 1장(Baseline)과 비교했을 때 출처의 신뢰성이 달라졌는가?
2. Agent가 검색을 몇 번 수행했는가? 충분했는가?
3. 검색 품질이 완벽하지 않을 수 있음 — 어떤 경우에 검색 결과가 부정확했는가?
4. 가능성은 낮지만 사례가 중복될 수 있음 — 중복을 방지하려면 Task 설명을 어떻게 바꿀 수 있는가?
5. 출처 신뢰성 편차가 있음 — 신뢰할 수 있는 출처와 그렇지 않은 출처를 구분하는 기준은 무엇인가?

👉 3장에서 동일한 주제로 SERPER(키워드 검색) vs EXA(의미 기반 검색)를 비교하면 어떤 차이가 나는지 확인해 보세요.